## Transform Drivers Data (Bronze → Silver)
Steps:
1. Read drivers.csv from Bronze
2. Add ingestion timestamp
3. Create driver_name column
4. Convert DOB to date format
5. Select & rename required columns
6. Register table in catalog
7. Store Delta files in ADLS

In [0]:
%run "/Workspace/f1-project/00-Configurations"

In [0]:
from pyspark.sql.functions import current_timestamp, concat, col, lit, to_date

In [0]:
drivers_df = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.option("nullValue", "\\N") \
.csv(f"{bronze_path}/drivers.csv")

In [0]:
drivers_df.show()

+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|driverId| driverRef|number|code| forename|   surname|       dob|nationality|                 url|
+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|       1|  hamilton|    44| HAM|    Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|
|       2|  heidfeld|  NULL| HEI|     Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|
|       3|   rosberg|     6| ROS|     Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|
|       4|    alonso|    14| ALO| Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|
|       5|kovalainen|  NULL| KOV|   Heikki|Kovalainen|1981-10-19|    Finnish|http://en.wikiped...|
|       6|  nakajima|  NULL| NAK|   Kazuki|  Nakajima|1985-01-11|   Japanese|http://en.wikiped...|
|       7|  bourdais|  NULL| BOU|Sébastien|  Bourdais|1979-02-28|     French|http://en.wikiped...|
|       8|

In [0]:
drivers_with_ingestion_df = drivers_df \
.withColumn("ingestion_date", current_timestamp())

In [0]:
drivers_with_name_df = drivers_with_ingestion_df \
.withColumn("driver_name", concat(col("forename"), lit(" "), col("surname")))

In [0]:
drivers_with_name_df.show()

+--------+----------+------+----+---------+----------+----------+-----------+--------------------+--------------------+------------------+
|driverId| driverRef|number|code| forename|   surname|       dob|nationality|                 url|      ingestion_date|       driver_name|
+--------+----------+------+----+---------+----------+----------+-----------+--------------------+--------------------+------------------+
|       1|  hamilton|    44| HAM|    Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|2026-03-14 11:09:...|    Lewis Hamilton|
|       2|  heidfeld|  NULL| HEI|     Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|2026-03-14 11:09:...|     Nick Heidfeld|
|       3|   rosberg|     6| ROS|     Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|2026-03-14 11:09:...|      Nico Rosberg|
|       4|    alonso|    14| ALO| Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|2026-03-14 11:09:...|   Fernando Alonso|
|       5|kovalainen|  NULL

In [0]:
drivers_transformed_df = drivers_with_name_df \
.withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))

In [0]:
drivers_selected_df = drivers_transformed_df.select(
    col("driverId").alias("driver_id"),
    col("driverRef").alias("driver_ref"),
    col("number"),
    col("code"),
    col("driver_name"),
    col("dob"),
    col("nationality"),
    col("ingestion_date")
)

In [0]:
drivers_selected_df.show()

+---------+----------+------+----+------------------+----------+-----------+--------------------+
|driver_id|driver_ref|number|code|       driver_name|       dob|nationality|      ingestion_date|
+---------+----------+------+----+------------------+----------+-----------+--------------------+
|        1|  hamilton|    44| HAM|    Lewis Hamilton|1985-01-07|    British|2026-03-14 11:09:...|
|        2|  heidfeld|  NULL| HEI|     Nick Heidfeld|1977-05-10|     German|2026-03-14 11:09:...|
|        3|   rosberg|     6| ROS|      Nico Rosberg|1985-06-27|     German|2026-03-14 11:09:...|
|        4|    alonso|    14| ALO|   Fernando Alonso|1981-07-29|    Spanish|2026-03-14 11:09:...|
|        5|kovalainen|  NULL| KOV| Heikki Kovalainen|1981-10-19|    Finnish|2026-03-14 11:09:...|
|        6|  nakajima|  NULL| NAK|   Kazuki Nakajima|1985-01-11|   Japanese|2026-03-14 11:09:...|
|        7|  bourdais|  NULL| BOU|Sébastien Bourdais|1979-02-28|     French|2026-03-14 11:09:...|
|        8| raikkone

In [0]:
drivers_selected_df.count()

864

In [0]:
drivers_final_df = drivers_selected_df

In [0]:
drivers_final_df = drivers_final_df.dropDuplicates(["driver_id"])

In [0]:
%sql
USE CATALOG f1_catalog

In [0]:
%sql
USE SCHEMA f1_silver

In [0]:
%sql
-- DROP TABLE drivers_1

In [0]:
drivers_final_df.write.mode("overwrite").format("delta").saveAsTable('drivers_1')


In [0]:
%sql
SELECT * FROM f1_silver.drivers_1

driver_id,driver_ref,number,code,driver_name,dob,nationality,ingestion_date
148,foitek,null,null,Gregor Foitek,1965-03-27,Swiss,2026-03-14T11:09:58.18231Z
463,chamberlain,null,null,Jay Chamberlain,1925-12-29,American,2026-03-14T11:09:58.18231Z
471,johnstone,null,null,Bruce Johnstone,1937-01-30,South African,2026-03-14T11:09:58.18231Z
496,menditeguy,null,null,Carlos Menditeguy,1914-08-10,Argentine,2026-03-14T11:09:58.18231Z
833,merhi,98,MER,Roberto Merhi,1991-03-22,Spanish,2026-03-14T11:09:58.18231Z
243,stommelen,null,null,Rolf Stommelen,1943-07-11,German,2026-03-14T11:09:58.18231Z
392,fisher,null,null,Mike Fisher,1943-03-13,American,2026-03-14T11:09:58.18231Z
540,mike_taylor,null,null,Mike Taylor,1934-04-24,British,2026-03-14T11:09:58.18231Z
623,uria,null,null,Alberto Uria,1924-07-11,Uruguayan,2026-03-14T11:09:58.18231Z
737,obrien,null,null,Robert O'Brien,1908-04-11,American,2026-03-14T11:09:58.18231Z


In [0]:
%sql
SELECT COUNT(*) FROM f1_silver.drivers_1

count(1)
864


In [0]:
drivers_final_df.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.save(f"{silver_path}/drivers")